Deep Reserach Agents

In [7]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [8]:
load_dotenv(override = True)

True

API based websearch tool

In [9]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

In [10]:
search_agent = Agent(
    name = "Search agent",
    instructions=INSTRUCTIONS,
    model='gpt-4o-mini',
    tools = [WebSearchTool(search_context_size="low")],
    model_settings=ModelSettings(tool_choice="required")
)

In [11]:
message = "Latest AI Agent frameworks in 2025"

In [12]:
with trace("Search"):
    result = await Runner.run(search_agent, message)

In [13]:
display(Markdown(result.final_output))

As of 2025, several AI agent frameworks have emerged, each offering unique capabilities for developing autonomous AI systems:

- **LangChain**: A modular framework with over 600 integrations, facilitating connections to major LLMs, tools, and databases. It supports retrieval-augmented generation, database queries, web scraping, and API calls. ([artificial-intelligence-wiki.com](https://artificial-intelligence-wiki.com/agentic-ai/agent-architectures-and-components/ai-agent-frameworks-guide/?utm_source=openai))

- **LangGraph**: An extension of LangChain, LangGraph introduces graph-based workflows for structured reasoning and state management, ideal for complex, branching LLM pipelines. ([codecademy.com](https://www.codecademy.com/article/top-ai-agent-frameworks-in-2025?utm_source=openai))

- **AutoGPT**: A goal-driven framework enabling autonomous experimentation, suitable for research and enterprise applications. ([codecademy.com](https://www.codecademy.com/article/top-ai-agent-frameworks-in-2025?utm_source=openai))

- **CrewAI**: Focuses on role-based, natural task splitting, facilitating multi-agent teamwork and rapid prototyping. ([codecademy.com](https://www.codecademy.com/article/top-ai-agent-frameworks-in-2025?utm_source=openai))

- **OpenAI Agents SDK**: Integrated into the OpenAI ecosystem, offering a simplified, API-first framework to connect LLMs with tools and memory, ideal for developers building applications within the OpenAI stack. ([codecademy.com](https://www.codecademy.com/article/top-ai-agent-frameworks-in-2025?utm_source=openai))

- **LightAgent**: A lightweight, open-source framework integrating memory, tools, and Tree of Thought (ToT) functionalities, designed for seamless integration with mainstream chat platforms. ([arxiv.org](https://arxiv.org/abs/2509.09292?utm_source=openai))

- **Codex**: Developed by OpenAI, Codex is an AI coding agent released in April 2025, designed for software engineering tasks such as writing code and fixing bugs. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Codex_%28AI_agent%29?utm_source=openai))

These frameworks cater to diverse needs, from general-purpose AI development to specialized applications in software engineering and multi-agent collaboration. 

Use Structured output and include description of fields

In [ ]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this

create websearch Item and websearchplan

In [17]:
class WebSearchItem(BaseModel):
    reason: str = Field(description ="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

In [18]:
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]= Field(description = "A list of web searches to perform to best answer the query.")

Create planner_agent and WebSerachPlan class to describe list of web searches, Search term used , resoning for search

In [19]:
planner_Agent = Agent(
    name = "PlannerAgent",
    instructions= INSTRUCTIONS, 
    model = 'gpt-4o-mini', 
    output_type= WebSearchPlan
)

In [20]:
message = "Latest AI Agent frameworks in 2025."

In [21]:
with trace("Search"):
    result = await Runner.run(planner_Agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find information on AI agent frameworks that will be trending or released in the year 2025.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To identify specific features and advancements in AI agent frameworks that are expected to emerge in 2025.', query='2025 AI agent frameworks features and advancements'), WebSearchItem(reason='To discover any major companies or research institutions focusing on developing AI agent frameworks in 2025.', query='AI agent frameworks companies research 2025')]


In [22]:
from sendgrid import from_email, to_email


@function_tool
def send_mail(subject:str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY')
    from_email = Email('neine112arora@gmail.com')
    to_email = To('neine112arora@gmail.com')
    content= Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    )

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3551860669.py, line 7)